<a href="https://colab.research.google.com/github/IbrahimAbdultwab/projects/blob/sp3t1/sp3t1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/twitterdataset/twitter_training.csv')
val_df = pd.read_csv('/content/drive/MyDrive/twitterdataset/twitter_validation.csv')
df = pd.concat([train_df, val_df], ignore_index=True)
df = df.iloc[:, :4]
df.columns = ['id', 'entity', 'sentiment', 'text']
df = df[['text', 'sentiment']]
df = df.dropna(subset=['text'])

In [ ]:
df = df[df['sentiment'].isin(['Positive', 'Negative', 'Neutral'])]
label_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df['label'] = df['sentiment'].map(label_map)

In [ ]:
X = df['clean_text']
y = df['label']

In [ ]:
max_words = 5000
max_len = 100
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X)
X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=max_len)

X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42)


In [ ]:
model = Sequential()
model.add(Embedding(max_words, 128, input_length=max_len))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(3, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=3, batch_size=64)


Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


747/747 ━━━━━━━━━━━━━━━━━━━━ 274s 359ms/step - accuracy: 0.6153 - loss: 0.8373 - val_accuracy: 0.7661 - val_loss: 0.5771
Epoch 2/3
747/747 ━━━━━━━━━━━━━━━━━━━━ 315s 350ms/step - accuracy: 0.8104 - loss: 0.4896 - val_accuracy: 0.8172 - val_loss: 0.4708
Epoch 3/3
747/747 ━━━━━━━━━━━━━━━━━━━━ 256s 342ms/step - accuracy: 0.8686 - loss: 0.3501 - val_accuracy: 0.8388 - val_loss: 0.4236


In [ ]:
model.save('sentiment_model.h5')

In [ ]:
from tensorflow.keras.models import load_model
import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save label map
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
with open('label_map.pkl', 'wb') as f:
    pickle.dump(label_map, f)

In [ ]:
from google.colab import files
files.download('sentiment_model.h5')
files.download('tokenizer.pkl')
files.download('label_map.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>